# Training protocol and saved-run inspection

Use this notebook to inspect maintained steady-flow and transient-drying configurations, their data roles, resolved objectives, stage dependencies, and saved runs. Inspection never starts training, loads model checkpoints, initializes tracking, or generates predictions.

Transient authored plans resolve through the maintained plan services. Matched A0/B/A1 previews show the common A0 origin and A1's dependency on B's measured train-time receipt; a planned budget is not evidence that a run completed. The active configuration and scientific defaults remain authoritative.

Run All performs configuration and metadata inspection only. Dataset references must already be published; missing references remain actionable loader errors. All launch commands below are Markdown examples submitted through Slurm. The optional complete data validation at the end is disabled by default.


## 2. Imports and environment

The setup imports configuration, installed notebook support, and display services only. It resolves unified storage paths without querying CUDA or accessing the generated-simulation area. Workflow controls are defined beside the operations they govern.

In [ ]:
from __future__ import annotations

import importlib.metadata
import platform
import shlex

import pandas as pd
from IPython.display import Markdown
from IPython.display import display as show

from src import common, experiments

PROJECT_ROOT = common.paths.get_project_root()
STORAGE_ROOT = common.paths.get_storage_root()
DATASET_ROOT = common.paths.get_datasets_root()
EXPERIMENTS_ROOT = common.paths.get_experiments_root()
PATH_DISPLAY_ROOTS = {
    "project_root": PROJECT_ROOT,
    "storage_root": STORAGE_ROOT,
}

TASK_CONFIG_ROOT = PROJECT_ROOT / "configs" / "03_learning"
_EXPERIMENT_PATHS = tuple(
    sorted(
        path
        for task_root in TASK_CONFIG_ROOT.iterdir()
        if task_root.is_dir()
        for path in (task_root / "experiments").rglob("*.yaml")
        if path.is_file()
    )
)
EXPERIMENT_CONFIGS = {path.relative_to(TASK_CONFIG_ROOT).with_suffix("").as_posix(): path for path in _EXPERIMENT_PATHS}
if not EXPERIMENT_CONFIGS:
    message = f"No maintained task-local experiment YAMLs found below {TASK_CONFIG_ROOT}"
    raise RuntimeError(message)

CONTEXT: experiments.notebook_support.NotebookContext | None = None


def package_version(distribution: str) -> str:
    """Return an installed distribution version or an unavailable marker."""
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


def show_tables(*tables: experiments.notebook_support.NotebookTable) -> None:
    """Display ordered support tables through the notebook's pandas surface."""
    for table in tables:
        if table.title is not None:
            show(Markdown(f"#### {table.title}"))
        show(pd.DataFrame(table.rows, columns=table.columns))


def require_context() -> experiments.notebook_support.NotebookContext:
    """Return the prepared context or raise one actionable prerequisite error."""
    if CONTEXT is None:
        message = "Training notebook context is unavailable. Run Step 5: Select, resolve, and validate one experiment."
        raise RuntimeError(message)
    return CONTEXT


In [ ]:
environment_table = pd.DataFrame(
    [
        ("Project root", "$PROJECT_ROOT", "Shared ICE repository root"),
        ("Storage root", "$STORAGE_ROOT", "Unified numbered scientific lifecycle root"),
        (
            "Dataset root",
            experiments.notebook_support.display_path(DATASET_ROOT, **PATH_DISPLAY_ROOTS),
            "Published dataset metadata and tensor payloads; not opened by this notebook",
        ),
        (
            "Experiments root",
            experiments.notebook_support.display_path(EXPERIMENTS_ROOT, **PATH_DISPLAY_ROOTS),
            "Run-path preview and optional summary inspection",
        ),
        ("Python", platform.python_version(), "Active notebook kernel"),
        ("Project package", package_version("grainlegumes-pino-drying"), "Installed distribution version"),
        ("pandas", package_version("pandas"), "Notebook table rendering"),
    ],
    columns=["Setting", "Value", "Meaning"],
)
show(environment_table)


## 3. Scientific data roles

The resolved YAML selects the Dataset and declares its Experiment membership. Steady-flow uses ID training and validation plus diagnostic OOD membership. Transient drying partitions cases into training, validation, and untouched ID test; its optional additional Evaluation datasets carry explicit role, relation, and assumptions. An empty additional dataset list is valid and provides no OOD evidence.

Only training membership fits preprocessing. Validation drives scheduling and checkpoint selection. ID test and additional Evaluation roles do not influence fitting or model selection. Persisted split and scaling identities govern resume, inference, and Evaluation. Small acceptance datasets establish execution contracts, not generalization.


## 4. Available maintained configurations

The catalog reads `configs/03_learning/<task>/experiments`. Select a recipe for the intended task and Dataset.


In [ ]:
available_experiments = pd.DataFrame(
    [
        (
            label,
            experiments.notebook_support.display_path(path, **PATH_DISPLAY_ROOTS),
            path.is_file(),
        )
        for label, path in EXPERIMENT_CONFIGS.items()
    ],
    columns=["Experiment", "Configuration", "Exists"],
)
show(available_experiments)

## 5. Select and resolve one protocol

Set `CONFIG_PATH` to a catalog entry or another explicit maintained configuration. `SELECTED_STAGE = None` previews the first stage; transient plans also accept `a0`, `b`, or `a1`. The first cell lists the built Datasets of that configuration's task; the second sets `TRAIN_DATASET` to one of them and, for a matched transient plan, `TRAIN_SECONDS` (Stage B's measured target; `None` keeps the configured value). They replace `data.train_dataset` in every stage and Stage B's `train_seconds`, exactly as `--train-dataset` and `--train-seconds` do for training; Step 10 shows the launch command with the same values. Resolving a plan displays all children but creates none.

The context resets before resolution. A task without a built Dataset stops the first cell. Missing Dataset references, a matched plan without Stage B's seconds, or invalid settings remain the original loader failure; publish the intended Dataset before retrying. A preview is not a completed Experiment or training result.


In [ ]:
CONFIG_PATH = next(iter(EXPERIMENT_CONFIGS.values()))  # Choose one configuration listed in Step 4

SELECTED_STAGE = None  # None, or an available comparison arm

CONTEXT = None
BUILT_DATASETS = experiments.notebook_support.list_built_datasets(CONFIG_PATH)
show(pd.DataFrame(BUILT_DATASETS, columns=["Built Dataset"]))

In [ ]:
TRAIN_DATASET = BUILT_DATASETS[0]  # Choose one Dataset listed above
TRAIN_SECONDS = None  # matched transient plans only: Stage-B training seconds, exactly like --train-seconds

CONTEXT = None
CONTEXT = experiments.notebook_support.prepare_notebook_context(
    CONFIG_PATH,
    stage=SELECTED_STAGE,
    train_dataset=TRAIN_DATASET,
    train_seconds=TRAIN_SECONDS,
)

context_status = pd.DataFrame(
    [
        (
            "Selected configuration",
            experiments.notebook_support.display_path(CONTEXT.config_path, **PATH_DISPLAY_ROOTS),
        ),
        ("Official run name", CONTEXT.official_config["run"]["name"]),
        (
            "Metadata roles validated",
            sum(preview.metadata_validated for preview in CONTEXT.dataset_previews),
        ),
    ],
    columns=["Preparation result", "Value"],
)
show(context_status)

In [ ]:
context = require_context()
show_tables(experiments.notebook_support.prepare_protocol_table(context))

additional = context.official_config["data"]["additional_evaluation_datasets"]
if not additional:
    show(Markdown("No additional Evaluation dataset is configured. No OOD result is implied."))

## 6. Resolved scientific and runtime settings

The tables show the selected task and architecture, data partition, optimization, loss, evaluation metric, and tracking policy. Transient settings also show sampling, time conditioning, scaling, curriculum, teacher handoff, and measured-budget dependencies. These are resolved values; notebook inspection does not change them.


In [ ]:
context = require_context()
configuration_tables = experiments.notebook_support.prepare_configuration_tables(context)
show_tables(*configuration_tables)

## 7. Dataset metadata and paths

The overview reads maintained manifest evidence without opening tensor payloads or rehashing their bytes. Counts and Dataset digests describe admitted metadata, not a fresh full-payload validation. Missing metadata is displayed as unavailable. Run the maintained execution preflight for actual data loading and backend admission.


In [ ]:
context = require_context()
dataset_table = experiments.notebook_support.prepare_dataset_table(
    context,
    **PATH_DISPLAY_ROOTS,
)
show_tables(dataset_table)

## 8. Run identity and output preview

The production configuration loader owns exact task, model, Dataset, seed, and stage identity. The displayed objective belongs to the selected task: steady-flow and transient drying have distinct metric contracts. Only validation selects checkpoints; an ID test result remains separate.

The intended run path is a pure preview. Inspection does not create directories, acquire writer locks, save a run config, or initialize W&B. Existing run directories require explicit maintained resume admission.


In [ ]:
context = require_context()
run_preview = experiments.notebook_support.prepare_run_preview_table(
    context,
    **PATH_DISPLAY_ROOTS,
)
show_tables(run_preview)

## 9. Execution prerequisites

Training uses the maintained Slurm ML launcher from the canonical repository checkout, with an already published Dataset and compatible environment. Confirm the inspected protocol, explicit case memberships, preprocessing policy, objective, seed, precision, and resource request before submitting it.

`<experiment_config>` is an authored direct or transient plan YAML; `<run_dir>` is an exact persisted run leaf. For a transient plan, the maintained CLI takes B's measured target with `--train-seconds` (unless the configuration sets it) and supports `--train-stages all` (A0, B, A1) and `--artifacts selected`, `none`, `all`, or a stage list. A1 consumes B's committed measured budget; missing teacher or receipt evidence must fail closed. This notebook does not fabricate them.

Optuna is a separate workflow and is not needed to inspect or execute an already selected pilot.


## 10. Slurm training

The command below carries the selection made in Step 5: `--train-dataset` is `TRAIN_DATASET`, `--train-seconds` is `TRAIN_SECONDS` when set, and a matched A0/B/A1 plan also selects its stages and artifacts explicitly. Replace `<experiment_config>` with the selected `CONFIG_PATH`; copy an appropriately bounded resource request from the canonical checkout.

Choose CPU, memory, and time for the actual protocol; examples do not authorize a campaign. CPU execution uses `--mode cpu --partition standard` without GPU GRES. The launcher supplies a strict device policy and returns job and log identities. No command in this section is executable notebook code.


In [ ]:
context = require_context()
train_command = (
    "./scripts/slurm_ml.sh --mode gpu --partition gpu --cpus-per-task 4 \\\n"
    "  --mem 32G --time 04:00:00 --gres gpu:rtx6000ada:1 train \\\n"
    f"  <experiment_config> --train-dataset {shlex.quote(TRAIN_DATASET)}"
)
if TRAIN_SECONDS is not None:
    train_command += f" --train-seconds {TRAIN_SECONDS}"
if context.protocol == "matched_a1_b":
    train_command += " --train-stages all --artifacts selected"
show(Markdown(f"```bash\n{train_command}\n```"))

## 11. Training health and saved evidence

Follow the launcher's exact log path:

```bash
tail -F <log_path>
```

Inspect loss, validation objective, learning rate, finite gradients, epoch completion, checkpoint publication, and any numerical or resource failure. Transient `history.json` retains completed-epoch evidence; matched B also publishes a measured training-budget receipt. Logs and summaries retain partial or failed execution. An epoch cap reached before a budget is consumed is not matched-budget success.

Tracking mode is defined by the YAML. Notebook inspection never authenticates or initializes W&B. Model accuracy, physical plausibility, and generalization remain distinct from successful execution.


## 12. Resume

Use the maintained launcher and an exact existing run directory, without a configuration; the stage's own saved `config.yaml` is read:

```bash
./scripts/slurm_ml.sh --mode gpu --partition gpu --cpus-per-task 4 \
  --mem 32G --time 04:00:00 --gres gpu:rtx6000ada:1 train --resume <stage_directory>
```

Resume admission verifies saved config, split, fitted preprocessing, checkpoint identity, and continuation state. Steady or direct epoch extension and matched partial-epoch continuation have task-specific rules. See `docs/04_training.md` and `docs/07_reproducibility_and_validation.md`; changing scientific settings is not a resume repair. `last_checkpoint.pt` owns continuation; `best_checkpoint.pt` owns selected-model inference and Evaluation.


## 13. Inspect a persisted run

Set `INSPECT_RUN_DIR` below to one actual run leaf. Inspection uses that run's saved configuration and summary, independently of the currently previewed recipe. File and directory existence is availability evidence only; it does not validate a checkpoint or declare artifacts ready.

Transient additional Evaluation paths derive from the saved dataset assignments. When no additional dataset is configured, no corresponding artifact is expected.


In [ ]:
INSPECT_RUN_DIR = None  # Exact persisted run path, or None to skip

context = require_context()
show_tables(experiments.notebook_support.prepare_run_output_inventory_table(task_id=context.task.id))

if INSPECT_RUN_DIR is None:
    show(Markdown("Run inspection skipped. Set `INSPECT_RUN_DIR` above to an actual run leaf."))
else:
    inspection = experiments.notebook_support.prepare_run_inspection(INSPECT_RUN_DIR)
    show_tables(*experiments.notebook_support.prepare_run_inspection_tables(inspection, **PATH_DISPLAY_ROOTS))

## 14. Persisted prediction artifacts and Evaluation

Open `04_eval_single_model.ipynb` to inspect one actual completed model. It loads admitted saved artifacts with reference and predicted fields, units, metrics, time support, and timing evidence. `05_eval_comparison_models.ipynb` requires genuinely available compatible runs. An absent second model or additional dataset leaves comparison unavailable.

Artifacts derive from the saved `best_checkpoint.pt`, split, fitted preprocessing, configuration, and actual Dataset support. For an exact persisted run, use the maintained launcher:

```bash
./scripts/slurm_ml.sh --mode cpu --partition standard --cpus-per-task 4 \
  --mem 16G --time 02:00:00 artifacts --run-dir "<run_dir>"
```

The artifact service performs task-specific admission and records device and spatial/temporal support. Use the exact saved run directory; a preview has no inference artifacts. Matched stage artifact selection is also available through the training command shown above. See `docs/05_evaluation.md` for artifact scope and comparison requirements.


## 15. Optional complete steady data validation

The maintained full-data notebook validator currently supports steady-flow. It reads and hashes complete data, verifies split coverage and training-only preprocessing, and can be expensive. It is disabled by default and must run in a Slurm allocation on ICE.

Transient data admission and backend, case partition, scaling, and loading checks belong to the maintained transient training/inference services. This section does not claim a steady validator has checked transient data.


In [ ]:
RUN_FULL_DATA_VALIDATION = False  # Explicit opt-in inside a suitable Slurm allocation

if not RUN_FULL_DATA_VALIDATION:
    show(Markdown("Full data validation skipped."))
else:
    context = require_context()
    if context.task.id != "steady_flow":
        message = "This full-data validator supports steady_flow. Use maintained transient execution admission."
        raise ValueError(message)
    import os

    if not os.environ.get("SLURM_JOB_ID"):
        message = "Run complete data validation within a suitable Slurm allocation on ICE."
        raise RuntimeError(message)
    full_validation = experiments.validation.data_pipeline.validate_full_data_pipeline(context.official_config)
    validation_presentation = experiments.notebook_support.prepare_validation_presentation(full_validation)
    show_tables(*validation_presentation.tables)
    show(Markdown(validation_presentation.conclusion))